# Capstone Notebook 02: Code-Aware Chunking, Dense Embeddings & FAISS Indexing
**Objective**: Experiment with recursive text chunking strategies (600 characters, 100 overlap), generate 384-dimensional dense vector embeddings using `all-MiniLM-L6-v2`, and build a persistent FAISS vector database.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.document_loader import DocumentLoader
from src.chunking import TechnicalChunker

# 1. Ingest Documents
loader = DocumentLoader('../data/documents')
docs = loader.load_all_documents()

# 2. Experiment with Technical Code-Aware Chunking
chunker = TechnicalChunker(chunk_size=600, chunk_overlap=100)
chunks = chunker.chunk_documents(docs)

print(f"Generated {len(chunks)} total text chunks from {len(docs)} document pages.")

In [ ]:
# Inspect Individual Chunk Payload Objects
for chk in chunks[:3]:
    meta = chk['metadata']
    print(f"[Chunk ID: {chk['chunk_id']} | File: {meta['source']} | Page: {meta['page']}]")
    print(f"Length: {meta['chunk_char_count']} characters")
    print("Chunk Text:\n" + "-" * 40)
    print(chk['content'])
    print("=" * 50)

In [ ]:
# Generate 384-Dimensional Embeddings & Build FAISS Vector Database
from src.embeddings import EmbeddingGenerator
from src.retriever import TechnicalRetriever

# Initialize Retriever & FAISS Index
retriever = TechnicalRetriever()
retriever.build_index(chunks)

# Save Vector Database to Disk
retriever.save('../vectorstore')
print("Vector store successfully built and saved to ../vectorstore/")

In [ ]:
# Test Top-K Semantic Similarity Search
test_query = "What is the endpoint and payload schema for charging payment intents?"
print(f"Query: '{test_query}'\n")

results = retriever.search(test_query, top_k=2)

for idx, match in enumerate(results, start=1):
    meta = match['metadata']
    print(f"Match #{idx} | Similarity Score: {match['similarity_score']}")
    print(f"Source: {meta['source']} (Page {meta['page']})")
    print("Retrieved Text Preview:\n" + match['content'][:250] + "\n" + "-" * 50)